# 11 — Buffer Sensitivity (DIAGNOSTIC)

> **This is a temporary diagnostic tool. It is NOT part of the final pipeline.**  
> **Scope: SpaceNet cities only** (raw building files available on Drive).  
> Do NOT remove or change the buffer based solely on these results.

## What the buffer does

**Location:** `src/metrics/vector/matching.py`, lines 27–29 (scalar path) and 84–86 (vectorised path)  
**Config key:** `vector.preprocessing.tau_buffer_m` (currently **2.0 m**)  

Both the reference and candidate geometries are expanded by `tau_buffer_m` metres  
**before** computing intersection/union for IoU.  
This tolerates small geo-registration offsets between datasets:  
two buildings 1 m apart (with buffer=2 m) will still overlap and potentially match.

**Effect:**  
- Inflates IoU values relative to the un-buffered geometry  
- Impact is larger for small buildings (buffer adds relatively more area)  
- Impact is smaller for large buildings (buffer adds relatively less area)

## What this notebook does

Re-runs tile-level IoU matching for SpaceNet cities at:  
- **buffer = 0 m** (no buffer)  
- **buffer = current value** (from config, default 2 m)  

Requires raw building files on Google Drive (`data/01_raw/{city}/vector/`).  
Cities where raw files are missing are skipped with a warning.

**Why SpaceNet only?** SpaceNet7 cities have well-curated reference data and are  
the primary benchmark context. They are also the cities most likely to have raw  
building files present on Drive.

**Output:** `outputs/scratch/buffer_sensitivity.csv`

In [ ]:
!pip install -q geopandas shapely
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load existing tile metrics (baseline = buffer 2 m) ───────
import sys, time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
# Code from GitHub, data from Drive (see colab_bootstrap.py in the repo).
# Drive PROJECT_ROOT/src is a stale hand-copy; never import from it.
import subprocess as _sp
_sp.run(['wget','-q','-O','/content/colab_bootstrap.py','https://raw.githubusercontent.com/GFDRR/urban_validation/raw/fix/pipeline-audit/colab_bootstrap.py'], check=False)
sys.path.insert(0, '/content')
from colab_bootstrap import setup as _setup
_setup(PROJECT_ROOT)

import yaml
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg.get('data_dir', 'data/01_raw')
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
SENS_DIR     = PROJECT_ROOT / 'outputs' / 'sensitivity_studies'
SENS_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

TILE_SENTINEL = 'vector_metrics_tiles_all_datasets.parquet'

vec_pre        = cfg['vector']['preprocessing']
TAU_BUFFER_M   = float(vec_pre.get('tau_buffer_m', 2.0))
TAU_OVERLAP    = float(vec_pre.get('iou_threshold', vec_pre.get('tau_overlap', 0.5)))
TAU_BOUNDARY_M = float(vec_pre.get('tau_boundary', 2.0))
MIN_AREA_M2    = float(vec_pre.get('min_area_m2', 20.0))
FIX_GEOMS      = bool(vec_pre.get('fix_invalid_geoms', True))

print(f'Pipeline buffer value : {TAU_BUFFER_M} m')
print(f'IoU threshold         : {TAU_OVERLAP}')
print(f'Data directory        : {DATA_DIR}')

# ── Load SpaceNet7 flag from AOI tracker ─────────────────────────────────────
TRACKER_PATH = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
ref_source_map = {}
ref_file_map   = {}  # city_id -> [reference_filename, ...]

if TRACKER_PATH.exists():
    tracker = pd.read_csv(TRACKER_PATH, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    id_col = 'dataset_folder_name'

    if 'reference_source' in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=[id_col])
            .set_index(id_col)['reference_source']
            .str.strip().str.lower()
            .to_dict()
        )

    ref_col = next(
        (c for c in tracker.columns if 'reference' in c.lower() and 'file' in c.lower()),
        None
    )
    print(f'Reference file column : {ref_col}')

    if ref_col:
        for _, row in tracker.dropna(subset=[id_col]).iterrows():
            city = str(row[id_col]).strip()
            raw = str(row.get(ref_col, '') or '')
            parts = [p.strip() for p in raw.split('|') if p.strip()]
            if parts:
                ref_file_map.setdefault(city, []).extend(parts)

    print(f'Cities with reference file info: {len(ref_file_map)}')
else:
    print(f'[WARN] Tracker not found at {TRACKER_PATH}')

# ── Restrict to SpaceNet cities ───────────────────────────────────────────────
# Buffer sensitivity is most meaningful for SpaceNet7 cities (well-curated
# reference data) and those are the cities most likely to have raw files on Drive.
CITY_SUBSET = sorted(c for c, src in ref_source_map.items() if src == 'spacenet')
if CITY_SUBSET:
    print(f'\nRunning for {len(CITY_SUBSET)} SpaceNet cities: {CITY_SUBSET}')
else:
    print('\n[WARN] No SpaceNet cities found in tracker — falling back to all cities.')
    CITY_SUBSET = None

# ── Load existing tile metrics (baseline: buffer = TAU_BUFFER_M) ──────────────
baseline_rows = []
city_dirs_all = sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL))

for city_dir in city_dirs_all:
    city = city_dir.name
    if CITY_SUBSET and city not in CITY_SUBSET:
        continue
    try:
        tile_df = pd.read_parquet(city_dir / TILE_SENTINEL)
        for ds, g in tile_df.groupby('dataset'):
            tp = int(g['tp'].sum()); fp = int(g['fp'].sum()); fn = int(g['fn'].sum())
            p  = tp / (tp + fp) if (tp + fp) else 0.0
            r  = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2*p*r / (p+r) if (p+r) else 0.0
            baseline_rows.append({'city': city, 'dataset': ds,
                                   'is_spacenet7': True,
                                   'f1_with_buffer': round(f1, 4),
                                   'n_ref': int(g['n_ref'].sum()),
                                   'n_cand': int(g['n_cand'].sum())})
    except Exception as e:
        print(f'  [WARN] {city}: {e}')

df_baseline = pd.DataFrame(baseline_rows)
print(f'\nBaseline loaded: {len(df_baseline)} city×dataset rows | {df_baseline["city"].nunique()} cities')
print('Cell 1 done.')

In [ ]:
# ── Cell 2 — Re-run IoU matching at buffer=0 and buffer=current ──────────────
#
# Loads raw reference + candidate buildings and tiles for each city from Drive,
# then runs match_buildings_iou() with tau_buffer_m=0 and tau_buffer_m=TAU_BUFFER_M.
# Cities where raw data is unavailable are skipped.
#
# This is computationally intensive (~2–10 min per city).
# Set CELL2_MAX_CITIES to limit the run, or set to None for all cities.

CELL2_MAX_CITIES = None  # set e.g. to 10 to test on first 10 cities

from src.metrics.vector.matching import match_buildings_iou
from src.utils.buildings import load_buildings
from src.utils.geometry import get_projected_crs
from src.utils.tiling import subset_by_tile


BUFFER_VALUES = [0.0, TAU_BUFFER_M]


def compute_f1_at_buffer(
    ref_all: gpd.GeoDataFrame,
    cand_all: gpd.GeoDataFrame,
    tiles: gpd.GeoDataFrame,
    tau_buffer: float,
    tau_overlap: float,
) -> dict:
    """Tile-level IoU matching at one buffer value; return city-level F1."""
    ref_sindex  = ref_all.sindex
    cand_sindex = cand_all.sindex
    tp_total = fp_total = fn_total = 0

    for tile_row in tiles.itertuples():
        ref_tile  = subset_by_tile(ref_all,  ref_sindex,  tile_row.geometry)
        cand_tile = subset_by_tile(cand_all, cand_sindex, tile_row.geometry)
        if ref_tile.empty and cand_tile.empty:
            continue
        matches_df, ref_unm, cand_unm = match_buildings_iou(
            ref_tile, cand_tile, tau_overlap, tau_buffer_m=tau_buffer
        )
        tp_total += len(matches_df)
        fp_total += len(cand_unm)
        fn_total += len(ref_unm)

    prec = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    rec  = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) else 0.0
    return {'tp': tp_total, 'fp': fp_total, 'fn': fn_total,
            'precision': round(prec, 4), 'recall': round(rec, 4), 'f1': round(f1, 4)}


def find_candidate_files(city: str, ds_name: str) -> list:
    city_slug = city.lower()
    vec_dir   = DATA_DIR / city / 'vector'
    pattern   = f'{city_slug.replace("-", "_")}_{ds_name}*.parquet'
    return sorted(vec_dir.glob(pattern))


def find_reference_files(city: str) -> list:
    if city in ref_file_map:
        paths = [DATA_DIR / city / 'vector' / fname
                 for fname in ref_file_map[city]]
        return [p for p in paths if p.exists()]
    # Fallback: glob for any gpkg/parquet in vector/ that isn't a candidate
    vec_dir = DATA_DIR / city / 'vector'
    if not vec_dir.exists():
        return []
    cand_names = {'overture', 'gba', 'globfp'}
    found = []
    for f in vec_dir.glob('*.gpkg'):
        if not any(n in f.stem.lower() for n in cand_names):
            found.append(f)
    return found


recomp_rows   = []
skipped_cities = []

cities_to_run = [
    row['city'] for _, row in df_baseline.drop_duplicates('city').iterrows()
]
if CELL2_MAX_CITIES:
    cities_to_run = cities_to_run[:CELL2_MAX_CITIES]

print(f'Re-running matching for {len(cities_to_run)} cities...')
print(f'  Buffer values: {BUFFER_VALUES}')
print(f'  IoU threshold: {TAU_OVERLAP}')
print()

for city in cities_to_run:
    t0 = time.time()
    ref_paths = find_reference_files(city)
    if not ref_paths:
        skipped_cities.append((city, 'reference files not found'))
        continue

    # Load tiles (projected CRS from saved GPKG)
    city_slug  = city.lower()
    tiles_path = DATA_DIR / city / 'tiles' / f'{city_slug}_tiles.gpkg'
    if not tiles_path.exists():
        skipped_cities.append((city, f'tiles not found at {tiles_path}'))
        continue
    try:
        tiles = gpd.read_file(tiles_path)
        crs   = tiles.crs.to_string()
    except Exception as e:
        skipped_cities.append((city, f'tiles load error: {e}'))
        continue

    # Load reference buildings
    try:
        ref_parts = [load_buildings(p, crs_work=crs,
                                    min_area_m2=MIN_AREA_M2,
                                    fix_invalid_geoms=FIX_GEOMS)
                     for p in ref_paths]
        ref_all = (gpd.GeoDataFrame(pd.concat(ref_parts, ignore_index=True),
                                    crs=ref_parts[0].crs)
                   if len(ref_parts) > 1 else ref_parts[0])
    except Exception as e:
        skipped_cities.append((city, f'reference load error: {e}'))
        continue

    # Per-dataset candidates
    ds_names = df_baseline[df_baseline['city'] == city]['dataset'].unique()
    any_ds_ran = False

    for ds_name in ds_names:
        cand_files = find_candidate_files(city, ds_name)
        if not cand_files:
            continue
        try:
            cand_all = load_buildings(cand_files[0], crs_work=crs,
                                      min_area_m2=MIN_AREA_M2,
                                      fix_invalid_geoms=FIX_GEOMS)
        except Exception as e:
            print(f'  [WARN] {city}/{ds_name}: candidate load error: {e}')
            continue

        row = {'city': city, 'dataset': ds_name,
               'is_spacenet7': (ref_source_map.get(city, 'other') == 'spacenet')}

        for buf in BUFFER_VALUES:
            res = compute_f1_at_buffer(ref_all, cand_all, tiles, buf, TAU_OVERLAP)
            suffix = 'no_buffer' if buf == 0.0 else f'buffer_{int(buf)}m'
            row[f'f1_{suffix}'] = res['f1']
            row[f'tp_{suffix}'] = res['tp']
        recomp_rows.append(row)
        any_ds_ran = True

    elapsed = time.time() - t0
    status = f'{len(ds_names)} datasets' if any_ds_ran else 'no candidate files'
    print(f'  {city:<40} {status}  ({elapsed:.0f}s)')

df_recomp = pd.DataFrame(recomp_rows)

if not df_recomp.empty:
    buf_col   = f'f1_buffer_{int(TAU_BUFFER_M)}m'
    nobuf_col = 'f1_no_buffer'
    if buf_col in df_recomp.columns and nobuf_col in df_recomp.columns:
        df_recomp['delta_buffer_effect'] = df_recomp[buf_col] - df_recomp[nobuf_col]

print(f'\nRecomputed: {len(df_recomp)} city×dataset rows')
if skipped_cities:
    print(f'Skipped: {len(skipped_cities)} cities')
    for city, reason in skipped_cities[:5]:
        print(f'  {city}: {reason}')
    if len(skipped_cities) > 5:
        print(f'  ... and {len(skipped_cities)-5} more')

In [ ]:
# ── Cell 3 — Comparison table + box plot ──────────────────────────────────────

if df_recomp.empty:
    print('[INFO] No recomputed data — run Cell 2 first (raw building files required).')
    print('       Showing distribution of existing IoU values from match parquets instead.')

    # Fallback: show IoU distribution from stored matches
    MATCH_SENTINEL = 'vector_matches_all_datasets.parquet'
    iou_parts = []
    for city_dir in sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL)):
        mp = city_dir / MATCH_SENTINEL
        if mp.exists():
            try:
                m = pd.read_parquet(mp, columns=['dataset', 'iou'])
                m['city'] = city_dir.name
                iou_parts.append(m)
            except Exception:
                pass

    if iou_parts:
        df_iou = pd.concat(iou_parts, ignore_index=True)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, ds in zip(axes, sorted(df_iou['dataset'].unique())[:2]):
            m = df_iou[df_iou['dataset'] == ds]['iou']
            ax.hist(m, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
            ax.axvline(0.5, color='tomato', linestyle='--', linewidth=1.5, label='τ=0.5')
            frac_near = ((m >= 0.45) & (m <= 0.55)).mean() * 100
            ax.set_title(f'{ds}\n{frac_near:.1f}% of matches near τ±0.05 (buffer-sensitive)')
            ax.set_xlabel('IoU (with buffer=2m applied)')
            ax.legend()
            sns.despine(ax=ax)
        fig.suptitle('Stored match IoU distribution — buffer-sensitive pairs are near τ=0.5',
                     fontweight='bold')
        fig.tight_layout()
        out_fig = SCRATCH_DIR / 'buffer_iou_distribution.png'
        fig.savefig(out_fig, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Figure saved → {out_fig}')
else:
    buf_col   = f'f1_buffer_{int(TAU_BUFFER_M)}m'
    nobuf_col = 'f1_no_buffer'

    # City-level averages
    df_city = (
        df_recomp
        .groupby(['city', 'is_spacenet7'])[[nobuf_col, buf_col, 'delta_buffer_effect']]
        .mean()
        .reset_index()
        .round(4)
    )
    df_city['group'] = df_city['is_spacenet7'].map(
        {True: 'SpaceNet7', False: 'Non-SpaceNet'}
    )

    GROUP_PALETTE = {True: '#0072B2', False: '#E69F00'}

    print('=== Summary by group ===')
    for group, grp in df_city.groupby('is_spacenet7'):
        label = 'SpaceNet7' if group else 'Non-SpaceNet'
        print(f'  {label} ({len(grp)} cities):')
        for col in [nobuf_col, buf_col, 'delta_buffer_effect']:
            if col in grp.columns:
                print(f'    {col:<35}: mean={grp[col].mean():.4f}  '
                      f'median={grp[col].median():.4f}')

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1: F1 with and without buffer
    df_melt = df_city.melt(
        id_vars=['city', 'group'],
        value_vars=[nobuf_col, buf_col],
        var_name='buffer', value_name='f1'
    )
    df_melt['buffer_label'] = df_melt['buffer'].map(
        {nobuf_col: 'buffer=0', buf_col: f'buffer={TAU_BUFFER_M}m'})
    sns.boxplot(data=df_melt, x='buffer_label', y='f1', hue='group',
                palette={'SpaceNet7': '#0072B2', 'Non-SpaceNet': '#E69F00'},
                ax=axes[0], linewidth=1.2)
    axes[0].set_title('F1 with vs without buffer', fontweight='bold')
    axes[0].set_xlabel('Buffer setting')
    axes[0].set_ylabel('City-level F1')
    axes[0].set_ylim(0, 1.05)
    axes[0].legend(title='Group', fontsize=8)

    # Panel 2: Delta (buffer effect = F1_with_buffer - F1_no_buffer)
    sns.boxplot(data=df_city, x='group', y='delta_buffer_effect',
                palette={'SpaceNet7': '#0072B2', 'Non-SpaceNet': '#E69F00'},
                ax=axes[1], linewidth=1.2)
    axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
    axes[1].set_title(f'Δ F1: buffer={TAU_BUFFER_M}m minus buffer=0\n(positive = buffer helps)',
                      fontweight='bold')
    axes[1].set_xlabel('Group')
    axes[1].set_ylabel('ΔF1')

    # Panel 3: Scatter F1_no_buffer vs F1_with_buffer
    for group_label, grp in df_recomp.groupby(
            df_recomp['is_spacenet7'].map({True: 'SpaceNet7', False: 'Non-SpaceNet'})):
        colour = '#0072B2' if group_label == 'SpaceNet7' else '#E69F00'
        axes[2].scatter(grp[nobuf_col], grp[buf_col],
                        color=colour, alpha=0.4, s=10, label=group_label)
    axes[2].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='y=x')
    axes[2].set_xlabel('F1 (buffer=0)')
    axes[2].set_ylabel(f'F1 (buffer={TAU_BUFFER_M}m)')
    axes[2].set_title('Scatter: F1 with vs without buffer\n(above y=x = buffer helped)',
                      fontweight='bold')
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.grid(axis='y', alpha=0.3)
        sns.despine(ax=ax)

    fig.suptitle(f'Buffer sensitivity — tau_buffer_m = {TAU_BUFFER_M} m  vs  0 m',
                 fontsize=13, fontweight='bold', y=1.01)
    fig.tight_layout()
    out_fig = SCRATCH_DIR / 'buffer_sensitivity_boxplot.png'
    fig.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved → {out_fig}')

In [ ]:
# ── Cell 4 — Save output CSV ──────────────────────────────────────────────────
out_path = SENS_DIR / 'buffer_sensitivity.csv'

buf_col   = f'f1_buffer_{int(TAU_BUFFER_M)}m'
nobuf_col = 'f1_no_buffer'

if df_recomp.empty:
    print('[INFO] df_recomp is empty — CSV will contain only existing baseline results.')
    df_out = df_baseline[['city', 'dataset', 'is_spacenet7',
                           'f1_with_buffer']].copy()
    df_out['f1_no_buffer'] = float('nan')
    df_out['delta_buffer_effect'] = float('nan')
    df_out['note'] = 'Recomputed F1 at buffer=0 not available — run Cell 2 with raw building data'
else:
    df_out = df_recomp.copy()
    df_out = df_out.rename(columns={buf_col: 'f1_with_buffer'})

df_out.to_csv(out_path, index=False)

print(f'Saved → {out_path}')
print(f'  Rows   : {len(df_out):,}')
print(f'  Columns: {list(df_out.columns)}')
if not df_recomp.empty and 'delta_buffer_effect' in df_out.columns:
    print()
    print('=== Buffer effect summary (Δ = F1_with_buffer - F1_no_buffer) ===')
    print(df_out.groupby('is_spacenet7')['delta_buffer_effect']
          .agg(['mean', 'median', 'std', 'count'])
          .round(4)
          .rename(index={True: 'SpaceNet7', False: 'Non-SpaceNet'})
          .to_string())